# Pokemon SQL — joins, aggregations & window functions

This notebook is the runnable companion to the **SQL & Databases** chapter. I do two things:

1. **Load my own Pokemon schema into an in-memory SQLite database**, seed it with a handful
   of rows, and run real joins, aggregations, and a window function against it. The schema
   comes from a MySQL Workbench ERD I built for a BYU-Idaho database assignment
   (`db_assignment/pokemon_erd.sql`).
2. **Run a few of my DS 250 baseball queries** against the real Lahman baseball SQLite
   database — *if* it's present on this machine (it's a 66 MB file I reference by absolute
   path and never copy into the repo).

### Adaptations from the original MySQL schema

The original `pokemon_erd.sql` is MySQL Workbench output and won't load into SQLite as-is.
Honest list of what I changed to make it run here:

- **Dropped MySQL-isms:** `ENGINE=InnoDB`, backtick identifiers, `VISIBLE` index qualifiers,
  and the `SET @OLD_UNIQUE_CHECKS=...` session lines — none exist in SQLite.
- **Fixed the `gym` duplicate-`location` bug** (SQLite rejects two columns of the same name)
  and dropped the empty stub tables `table1`/`table2`.
- **Corrected the `region` foreign key direction** — put `region_id` on `pokemon` instead of
  `pokemon_id` on `region` (a region has many pokemon, not the reverse).
- **Added name columns** to `generation` (the original had only an id) so the join results
  are human-readable, matching the intent shown in the original `Queries.txt`.
- Loaded a **focused subset** of the 18 tables — enough to demonstrate every relationship
  type without transcribing the whole schema.

In [1]:
import sqlite3
import pandas as pd

# In-memory database: nothing is written to disk, so this notebook is self-contained.
con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON;")
print("SQLite version:", sqlite3.sqlite_version)  # window functions need >= 3.25

SQLite version: 3.39.4


## Build the schema

A focused subset of my Pokemon ERD, adapted to SQLite. Note the three relationship shapes:
one-to-many (`pokemon.generation_id`), the ability hub (`abilites` pointing at three ability
tables), and two many-to-many junction tables (`pokemon_has_type`, `trainer_has_pokemon`).

In [2]:
schema = """
CREATE TABLE generation (
    generation_id  INTEGER PRIMARY KEY,
    generation     TEXT NOT NULL
);
CREATE TABLE region (
    region_id      INTEGER PRIMARY KEY,
    region_name    TEXT NOT NULL
);
CREATE TABLE rarity (
    rarity_id      INTEGER PRIMARY KEY,
    rarity_name    TEXT NOT NULL
);
CREATE TABLE first_ability  (f_ability_id      INTEGER PRIMARY KEY, ability_name TEXT);
CREATE TABLE second_ability (second_ability_id INTEGER PRIMARY KEY, ability_name TEXT);
CREATE TABLE hidden_ability (hidden_ability_id INTEGER PRIMARY KEY, ability_name TEXT);
CREATE TABLE abilites (
    abilites_id        INTEGER PRIMARY KEY,
    f_ability_id       INTEGER REFERENCES first_ability(f_ability_id),
    second_ability_id  INTEGER REFERENCES second_ability(second_ability_id),
    hidden_ability_id  INTEGER REFERENCES hidden_ability(hidden_ability_id)
);
CREATE TABLE type (
    type_id   INTEGER PRIMARY KEY,
    name      TEXT NOT NULL
);
CREATE TABLE pokemon (
    pokemon_id      INTEGER PRIMARY KEY,
    pokemon_name    TEXT NOT NULL,
    pokemon_weight  REAL,          -- kg
    pokemon_height  REAL,          -- m
    generation_id   INTEGER REFERENCES generation(generation_id),
    region_id       INTEGER REFERENCES region(region_id),
    rarity_id       INTEGER REFERENCES rarity(rarity_id),
    abilites_id     INTEGER REFERENCES abilites(abilites_id)
);
CREATE TABLE pokemon_has_type (
    pokemon_id  INTEGER REFERENCES pokemon(pokemon_id),
    type_id     INTEGER REFERENCES type(type_id),
    PRIMARY KEY (pokemon_id, type_id)
);
CREATE TABLE trainer (
    trainer_id  INTEGER PRIMARY KEY,
    first_name  TEXT,
    last_name   TEXT
);
CREATE TABLE trainer_has_pokemon (
    trainer_id  INTEGER REFERENCES trainer(trainer_id),
    pokemon_id  INTEGER REFERENCES pokemon(pokemon_id),
    PRIMARY KEY (trainer_id, pokemon_id)
);
"""
con.executescript(schema)
print("Tables:", [r[0] for r in con.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])

Tables: ['abilites', 'first_ability', 'generation', 'hidden_ability', 'pokemon', 'pokemon_has_type', 'rarity', 'region', 'second_ability', 'trainer', 'trainer_has_pokemon', 'type']


## Seed a few rows

The region / generation / rarity values are lifted straight from my assignment's
`Stuff for Insert Into.txt`; the pokemon rows are a small hand-picked set with roughly
real weights (kg) and heights (m).

In [3]:
cur = con.cursor()

cur.executemany("INSERT INTO generation VALUES (?, ?)", [
    (1,"Generation I"),(2,"Generation II"),(3,"Generation III"),
    (4,"Generation IV"),(5,"Generation V")])

cur.executemany("INSERT INTO region VALUES (?, ?)", [
    (1,"Kanto"),(2,"Johto"),(3,"Hoenn"),(4,"Sinnoh"),(5,"Unova")])

cur.executemany("INSERT INTO rarity VALUES (?, ?)", [
    (1,"Common"),(2,"Uncommon"),(3,"Rare"),(4,"Legendary"),(5,"Mythic"),(6,"Starter")])

cur.executemany("INSERT INTO type VALUES (?, ?)", [
    (1,"Grass"),(2,"Fire"),(3,"Water"),(4,"Electric"),(5,"Poison"),
    (6,"Flying"),(7,"Psychic"),(8,"Normal"),(9,"Ghost"),(10,"Dragon")])

cur.executemany("INSERT INTO first_ability VALUES (?, ?)", [
    (1,"Overgrow"),(2,"Blaze"),(3,"Torrent"),(4,"Static"),
    (5,"Pressure"),(6,"Cursed Body"),(7,"Immunity")])
cur.executemany("INSERT INTO second_ability VALUES (?, ?)", [
    (1,"Chlorophyll"),(2,"Solar Power"),(3,"Rain Dish"),(4,"Lightning Rod")])
cur.executemany("INSERT INTO hidden_ability VALUES (?, ?)", [
    (1,"Unnerve"),(2,"Thick Fat")])

# ability sets: (set_id, first, second, hidden) -- second/hidden may be NULL
cur.executemany("INSERT INTO abilites VALUES (?, ?, ?, ?)", [
    (1,1,1,None),(2,2,2,None),(3,3,3,None),(4,4,4,None),
    (5,5,None,1),(6,6,None,None),(7,7,None,2)])

# pokemon: id, name, weight_kg, height_m, generation, region, rarity, ability_set
cur.executemany("INSERT INTO pokemon VALUES (?, ?, ?, ?, ?, ?, ?, ?)", [
    (1,"Bulbasaur",   6.9, 0.7, 1, 1, 6, 1),
    (4,"Charmander",  8.5, 0.6, 1, 1, 6, 2),
    (7,"Squirtle",    9.0, 0.5, 1, 1, 6, 3),
    (6,"Charizard",  90.5, 1.7, 1, 1, 3, 2),
    (25,"Pikachu",    6.0, 0.4, 1, 1, 2, 4),
    (94,"Gengar",    40.5, 1.5, 1, 1, 3, 6),
    (143,"Snorlax", 460.0, 2.1, 1, 1, 3, 7),
    (150,"Mewtwo",   122.0, 2.0, 1, 1, 4, 5),
    (152,"Chikorita", 6.4, 0.9, 2, 2, 6, 1),
    (258,"Mudkip",    7.6, 0.4, 3, 3, 6, 3),
    (448,"Lucario",  54.0, 1.2, 4, 4, 3, 6)])

cur.executemany("INSERT INTO pokemon_has_type VALUES (?, ?)", [
    (1,1),(1,5),        # Bulbasaur: Grass/Poison
    (4,2),(7,3),        # Charmander Fire, Squirtle Water
    (6,2),(6,6),        # Charizard: Fire/Flying
    (25,4),             # Pikachu Electric
    (94,9),(94,5),      # Gengar: Ghost/Poison
    (143,8),            # Snorlax Normal
    (150,7),            # Mewtwo Psychic
    (152,1),(258,3),(448,10)])

cur.executemany("INSERT INTO trainer VALUES (?, ?, ?)", [
    (1,"Ash","Ketchum"),(2,"Misty",None),(3,"Brock",None)])
cur.executemany("INSERT INTO trainer_has_pokemon VALUES (?, ?)", [
    (1,25),(1,6),(1,143),(2,7),(3,94)])

con.commit()
print("pokemon rows:", con.execute("SELECT COUNT(*) FROM pokemon").fetchone()[0])

pokemon rows: 11


## Join — flatten foreign keys into readable values

This is the "master key" pattern from my assignment's `Queries.txt`: chain `LEFT JOIN`s so
each foreign key on `pokemon` becomes a human-readable value, and a pokemon still appears
even where a related row is missing.

In [4]:
pd.read_sql_query("""
    SELECT p.pokemon_name, p.pokemon_weight, p.pokemon_height,
           g.generation, r.region_name, rar.rarity_name,
           fa.ability_name AS first_ability
    FROM pokemon p
    LEFT JOIN generation g   ON p.generation_id = g.generation_id
    LEFT JOIN region     r   ON p.region_id     = r.region_id
    LEFT JOIN rarity     rar ON p.rarity_id      = rar.rarity_id
    LEFT JOIN abilites   a   ON p.abilites_id    = a.abilites_id
    LEFT JOIN first_ability fa ON a.f_ability_id = fa.f_ability_id
    ORDER BY p.pokemon_id
""", con)

,pokemon_name,pokemon_weight,pokemon_height,generation,region_name,rarity_name,first_ability
0,Bulbasaur,6.9,0.7,Generation I,Kanto,Starter,Overgrow
1,Charmander,8.5,0.6,Generation I,Kanto,Starter,Blaze
2,Charizard,90.5,1.7,Generation I,Kanto,Rare,Blaze
3,Squirtle,9.0,0.5,Generation I,Kanto,Starter,Torrent
4,Pikachu,6.0,0.4,Generation I,Kanto,Uncommon,Static
5,Gengar,40.5,1.5,Generation I,Kanto,Rare,Cursed Body
6,Snorlax,460.0,2.1,Generation I,Kanto,Rare,Immunity
7,Mewtwo,122.0,2.0,Generation I,Kanto,Legendary,Pressure
8,Chikorita,6.4,0.9,Generation II,Johto,Starter,Overgrow
9,Mudkip,7.6,0.4,Generation III,Hoenn,Starter,Torrent


## Many-to-many — resolve types through the junction table

A pokemon has several types and a type has many pokemon, so the relationship lives in the
`pokemon_has_type` junction table. Joining *through* it, then `GROUP_CONCAT`-ing, gives one
row per pokemon with its type list.

In [5]:
pd.read_sql_query("""
    SELECT p.pokemon_name,
           GROUP_CONCAT(t.name, '/') AS types,
           COUNT(t.type_id) AS num_types
    FROM pokemon p
    JOIN pokemon_has_type pt ON p.pokemon_id = pt.pokemon_id
    JOIN type t              ON pt.type_id   = t.type_id
    GROUP BY p.pokemon_id, p.pokemon_name
    ORDER BY num_types DESC, p.pokemon_name
""", con)

,pokemon_name,types,num_types
0,Bulbasaur,Grass/Poison,2
1,Charizard,Fire/Flying,2
2,Gengar,Ghost/Poison,2
3,Charmander,Fire,1
4,Chikorita,Grass,1
5,Lucario,Dragon,1
6,Mewtwo,Psychic,1
7,Mudkip,Water,1
8,Pikachu,Electric,1
9,Snorlax,Normal,1


## Aggregation — count and average per group

`GROUP BY` collapses rows into one summary per group. Here: how many pokemon and their
average weight, per type. `HAVING` would filter these groups (e.g. only types with 2+
pokemon) — `WHERE` can't, because it runs before the grouping.

In [6]:
pd.read_sql_query("""
    SELECT t.name AS type,
           COUNT(*)                    AS num_pokemon,
           ROUND(AVG(p.pokemon_weight), 1) AS avg_weight_kg
    FROM type t
    JOIN pokemon_has_type pt ON t.type_id = pt.type_id
    JOIN pokemon p           ON pt.pokemon_id = p.pokemon_id
    GROUP BY t.type_id, t.name
    ORDER BY num_pokemon DESC, type
""", con)

,type,num_pokemon,avg_weight_kg
0,Fire,2,49.5
1,Grass,2,6.7
2,Poison,2,23.7
3,Water,2,8.3
4,Dragon,1,54.0
5,Electric,1,6.0
6,Flying,1,90.5
7,Ghost,1,40.5
8,Normal,1,460.0
9,Psychic,1,122.0


## Window function — rank without collapsing rows

A window function keeps every row **and** computes across a related set. Here `RANK() OVER
(PARTITION BY generation ORDER BY weight DESC)` ranks each pokemon by weight *within its
generation* — every pokemon stays in the result, each labeled with its heaviest-first rank.
(SQLite has supported window functions since 3.25.)

In [7]:
pd.read_sql_query("""
    SELECT p.pokemon_name, g.generation, p.pokemon_weight,
           RANK() OVER (
               PARTITION BY p.generation_id
               ORDER BY p.pokemon_weight DESC
           ) AS weight_rank_in_gen
    FROM pokemon p
    JOIN generation g ON p.generation_id = g.generation_id
    ORDER BY p.generation_id, weight_rank_in_gen
""", con)

,pokemon_name,generation,pokemon_weight,weight_rank_in_gen
0,Snorlax,Generation I,460.0,1
1,Mewtwo,Generation I,122.0,2
2,Charizard,Generation I,90.5,3
3,Gengar,Generation I,40.5,4
4,Squirtle,Generation I,9.0,5
5,Charmander,Generation I,8.5,6
6,Bulbasaur,Generation I,6.9,7
7,Pikachu,Generation I,6.0,8
8,Chikorita,Generation II,6.4,1
9,Mudkip,Generation III,7.6,1


---

## Part 2 — real baseball queries against the Lahman database

These are queries from my **DS 250 Project 3** (BYU-Idaho), run against the real Lahman
baseball SQLite database. That file is ~66 MB, so I reference it by **absolute path** and
never copy it into the repo. If it isn't present on this machine, this section is skipped
gracefully and the committed outputs above still tell the story.

In [8]:
import os

LAHMAN = "/Users/tylerlowry/Projects/school/byui-undergrad/DS_250/Project 3/lahmansbaseballdb.sqlite"
have_lahman = os.path.exists(LAHMAN)
print("Lahman DB found:", have_lahman)
bcon = sqlite3.connect(LAHMAN) if have_lahman else None

Lahman DB found: True


**Q1 — BYU-Idaho players who reached the majors, and their salaries.** A three-table
join: school -> who played there -> their salaries.

In [9]:
if bcon:
    display(pd.read_sql_query("""
        SELECT c.playerID, s.schoolID, sa.salary, sa.yearid, sa.teamid
        FROM schools s
        JOIN collegeplaying c ON c.schoolid = s.schoolid
        JOIN salaries sa      ON sa.playerid = c.playerid
        WHERE s.schoolid = 'idbyuid'
        ORDER BY sa.salary DESC
        LIMIT 10
    """, bcon))
else:
    print("Skipped — Lahman DB not on this machine.")

,playerID,schoolID,salary,yearID,teamID
0,lindsma01,idbyuid,4000000.0,2014,CHA
1,lindsma01,idbyuid,4000000.0,2014,CHA
2,lindsma01,idbyuid,3600000.0,2012,BAL
3,lindsma01,idbyuid,3600000.0,2012,BAL
4,lindsma01,idbyuid,2800000.0,2011,COL
5,lindsma01,idbyuid,2800000.0,2011,COL
6,lindsma01,idbyuid,2300000.0,2013,CHA
7,lindsma01,idbyuid,2300000.0,2013,CHA
8,lindsma01,idbyuid,1625000.0,2010,HOU
9,lindsma01,idbyuid,1625000.0,2010,HOU


**Q2 — top career batting averages** (min 100 at-bats), showing the `CAST`-to-`FLOAT`
fix for integer division plus `GROUP BY`/`AVG`.

In [10]:
if bcon:
    display(pd.read_sql_query("""
        SELECT b.playerid,
               ROUND(AVG(CAST(b.h AS FLOAT)/CAST(b.ab AS FLOAT)), 3) AS batting_avg
        FROM batting b
        WHERE b.ab > 100
        GROUP BY b.playerid
        ORDER BY batting_avg DESC, b.playerid
        LIMIT 5
    """, bcon))
else:
    print("Skipped — Lahman DB not on this machine.")

,playerID,batting_avg
0,hazlebo01,0.403
1,daviscu01,0.381
2,fishesh01,0.374
3,woltery01,0.370
4,barnero01,0.367


**Q3 — Cubs vs. White Sox all-time winning percentage** (my Project 3 team comparison),
summarized to one row per franchise instead of the full per-year series.

In [11]:
if bcon:
    display(pd.read_sql_query("""
        SELECT f.franchname,
               ROUND(AVG(CAST(t.w AS FLOAT)/CAST(t.g AS FLOAT)), 3) AS avg_win_pct,
               MIN(t.yearid) AS from_year, MAX(t.yearid) AS to_year
        FROM teamsfranchises f
        JOIN teams t ON t.franchid = f.franchid
        WHERE f.franchid IN ('CHC', 'CHW')
        GROUP BY f.franchid, f.franchname
    """, bcon))
    bcon.close()
else:
    print("Skipped — Lahman DB not on this machine.")

con.close()

,franchName,avg_win_pct,from_year,to_year
0,Chicago Cubs,0.514,1876,2019
1,Chicago White Sox,0.499,1901,2019


### Takeaways

- The Pokemon half is fully self-contained (in-memory SQLite, my own schema) — it runs
  anywhere, no data files, no license issues.
- The baseball half runs against my real DS 250 database when present, and degrades cleanly
  when it isn't, so the committed notebook renders either way.
- Same SQL toolkit throughout: joins to reassemble normalized data, `GROUP BY`/`AVG` to
  summarize, and window functions to rank without collapsing rows.